# Running `vllm` as a server with a smaller model

If you want to remove memory pressure from your GPU
(or have more cache memory available), you can work
with a quantized model.

Luckily, Google provides these models (which have been
specially trained) directly

We will use the OpenAI client:

In [ ]:
from openai import OpenAI

An API key can be added, but we don't need it here

In [ ]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

Run: `vllm serve google/gemma-4-12B-it-qat-w4a16-ct --max-model-len 8192 --tensor-parallel-size 1 --enable-auto-tool-choice --tool-call-parser gemma4 --reasoning-parser gemma4 --gpu-memory-utilization 0.92`

The chat completion can be used in the usual way:

In [ ]:
model = "google/gemma-4-12B-it-qat-w4a16-ct"

In [ ]:
%%time
completion = client.chat.completions.create(
    model=model, 
    messages=[{ "role": "user",
                "content": "Explain O'Reilly online learning!" } ]
)

Note the speed of the generation (in the terminal window) which is much faster!

In [ ]:
completion

In [ ]:
completion.choices[0].message.content

In [ ]:
from IPython.display import display, Markdown
display(Markdown(completion.choices[0].message.content))

It is also possible to work with the more modern `responses` API. 
Here we change the request to see a bit more the generation speed
in the console.

In [ ]:
response = client.responses.create(model=model, 
                                   input="Explain O'Reilly online learning! Give a very long answer!")

In [ ]:
response

In [ ]:
display(Markdown(response.output_text))

Now, take a look at the GPU memory usage.

In [ ]:
!nvidia-smi

As you can see, `vllm` uses again *all* memory and tries to optimize the generation process with caches.